In [ ]:
import os
import glob
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from datetime import datetime

# ─────────────────────────────────────────────
# 1. DATABASE CONNECTION — update if needed
# ─────────────────────────────────────────────
DB_CONFIG = {
    "host":     "localhost",
    "port":     5432,
    "database": "nhs_ae_analytics",
    "user":     "postgres",
    "password": "nhs1234"   # <-- change this
}

# ─────────────────────────────────────────────
# 2. FOLDER PATHS — your 4 year folders
# ─────────────────────────────────────────────
BASE_PATH = r"E:\Data Base\Fiverr project\PowerBI\Vivek"

YEAR_FOLDERS = [
    os.path.join(BASE_PATH, "2022-23"),
    os.path.join(BASE_PATH, "2023-24"),
    os.path.join(BASE_PATH, "2024-25"),
    os.path.join(BASE_PATH, "2025-26"),
]

# ─────────────────────────────────────────────
# 3. COLUMN POSITIONS IN THE EXCEL FILE
#    (positions 1-27 after skipping blank col 0)
# ─────────────────────────────────────────────
COL_POSITIONS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15,
                 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]

COL_NAMES = [
    "org_code",
    "region",
    "org_name",
    "type1_attendances",
    "type2_attendances",
    "type3_attendances",
    "total_attendances",
    "type1_under4hr",
    "type2_under4hr",
    "type3_under4hr",
    "total_under4hr",
    "type1_over4hr",
    "type2_over4hr",
    "type3_over4hr",
    "total_over4hr",
    "pct_within4hr_all",
    "pct_within4hr_type1",
    "pct_within4hr_type2",
    "pct_within4hr_type3",
    "emergency_admissions_type1",
    "emergency_admissions_type2",
    "emergency_admissions_type3",
    "total_emergency_admissions_via_ae",
    "other_emergency_admissions",
    "total_emergency_admissions",
    "over4hr_decision_to_admit",
    "over12hr_decision_to_admit",
]

# ─────────────────────────────────────────────
# 4. HELPER: extract period and financial year
# ─────────────────────────────────────────────
def get_period_info(filepath):
    """
    Row 5 of the file contains the period: e.g. 'April 2022'
    We read it and derive the NHS financial year.
    """
    df_meta = pd.read_excel(
        filepath,
        sheet_name="Provider Level Data",
        header=None,
        nrows=10
    )
    # Period label is in column 2, row index 5
    period_str = str(df_meta.iloc[5, 2]).strip()  # e.g. "April 2022"

    try:
        period_date = datetime.strptime(period_str, "%B %Y")
    except ValueError:
        # Fallback: try to find it by scanning meta rows
        for i in range(10):
            for j in range(5):
                val = str(df_meta.iloc[i, j])
                if "Period:" in val or "Period" in val:
                    period_str = str(df_meta.iloc[i, 2]).strip()
                    period_date = datetime.strptime(period_str, "%B %Y")
                    break

    month = period_date.month
    year  = period_date.year

    # NHS financial year: April = start of new year
    if month >= 4:
        fin_year = f"{year}-{str(year + 1)[2:]}"   # e.g. 2022-23
    else:
        fin_year = f"{year - 1}-{str(year)[2:]}"   # e.g. 2022-23

    return period_str, fin_year


# ─────────────────────────────────────────────
# 5. HELPER: read one Excel file → DataFrame
# ─────────────────────────────────────────────
def read_nhs_file(filepath):
    """
    Reads the Provider Level Data sheet from one NHS monthly file.
    Skips the 17 metadata/header rows, extracts Trust rows only.
    Returns a clean DataFrame ready for staging.
    """
    period_str, fin_year = get_period_info(filepath)

    # Read from row 18 onwards (skip rows 0–17 = 17 header rows + 1 blank)
    # No header — we assign column names manually
    df = pd.read_excel(
        filepath,
        sheet_name="Provider Level Data",
        header=None,
        skiprows=17
    )

    # Select only the columns we need by position
    df = df.iloc[:, COL_POSITIONS].copy()
    df.columns = COL_NAMES

    # Convert all columns to string to safely handle *, -, numbers
    df = df.astype(str)

    # Remove the England total row (org_code = "-")
    df = df[df["org_code"] != "-"]

    # Remove blank rows (org_code is NaN / "nan")
    df = df[~df["org_code"].isin(["nan", "NaN", ""])]

    # Remove rows where org_name is missing (footer rows at bottom of sheet)
    df = df[~df["org_name"].isin(["nan", "NaN", ""])]

    # Clean up "nan" strings in all columns → empty string
    # (PostgreSQL will store these as NULL via our staging SQL)
    df = df.replace("nan", "")
    df = df.replace("NaN", "")

    # Add period columns
    df.insert(0, "file_month",     period_str)   # e.g. "April 2022"
    df.insert(1, "financial_year", fin_year)      # e.g. "2022-23"

    return df


# ─────────────────────────────────────────────
# 6. MAIN: load all files into PostgreSQL
# ─────────────────────────────────────────────
def main():
    # Find all .xls and .xlsx files across all year folders
    all_files = []
    for folder in YEAR_FOLDERS:
        if not os.path.exists(folder):
            print(f"  WARNING: folder not found — {folder}")
            continue
        files = glob.glob(os.path.join(folder, "*.xls"))
        files += glob.glob(os.path.join(folder, "*.xlsx"))
        all_files.extend(files)

    if not all_files:
        print("ERROR: No files found. Check your BASE_PATH and folder names.")
        return

    print(f"Found {len(all_files)} files to load.\n")

    # Connect to PostgreSQL
    conn = psycopg2.connect(**DB_CONFIG)
    cur  = conn.cursor()

    # Clear any existing staging data (so we can re-run safely)
    cur.execute("TRUNCATE TABLE stg_ae_raw;")
    print("Staging table cleared. Loading files...\n")

    total_rows = 0

    for filepath in sorted(all_files):
        filename = os.path.basename(filepath)
        try:
            df = read_nhs_file(filepath)
            rows = [tuple(row) for row in df.values]

            insert_sql = """
                INSERT INTO stg_ae_raw (
                    file_month, financial_year,
                    org_code, region, org_name,
                    type1_attendances, type2_attendances, type3_attendances, total_attendances,
                    type1_under4hr, type2_under4hr, type3_under4hr, total_under4hr,
                    type1_over4hr, type2_over4hr, type3_over4hr, total_over4hr,
                    pct_within4hr_all, pct_within4hr_type1, pct_within4hr_type2, pct_within4hr_type3,
                    emergency_admissions_type1, emergency_admissions_type2, emergency_admissions_type3,
                    total_emergency_admissions_via_ae, other_emergency_admissions, total_emergency_admissions,
                    over4hr_decision_to_admit, over12hr_decision_to_admit
                ) VALUES %s
            """
            execute_values(cur, insert_sql, rows)
            conn.commit()

            print(f"  OK  {filename:55s}  →  {len(df):3d} rows")
            total_rows += len(df)

        except Exception as e:
            conn.rollback()
            print(f"  FAIL {filename}  →  {e}")

    cur.close()
    conn.close()

    print(f"\nDone. Total rows loaded: {total_rows}")
    print("Expected: roughly 200 rows × number of files loaded")
    print("\nNext step: run data profiling queries in pgAdmin.")


if __name__ == "__main__":
    main()